# Training SmolVLA

## Getting the Repo
---

In [1]:
!git clone https://github.com/Minko82/xlerobot-pro

Cloning into 'xlerobot-pro'...
remote: Enumerating objects: 15583, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 15583 (delta 1), reused 4 (delta 1), pack-reused 15579 (from 2)
Receiving objects: 100% (15583/15583), 230.21 MiB | 22.50 MiB/s, done.
Resolving deltas: 100% (7890/7890), done.


In [1]:
%cd xlerobot-pro/

/content/xlerobot-pro


## Getting the right deps
---
NOTE: Running these cells sequentially, you might run into an error telling you that you need to restart your runtime. That is fine, in that case, just run all cells sequentially starting from the `cd` cell.

In [2]:
!pip install -e '.[smolvla]'


Obtaining file:///content/xlerobot-pro
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for lerobot (pyproject.toml) ... done
  Created wheel for lerobot: filename=lerobot-0.4.1-0.editable-py3-none-any.whl size=10297 sha256=fc837b7320eeb696c9475c7f1cafb3d5aac69d0725b957f8cae04dc9861a8200
  Stored in directory: /tmp/pip-ephem-wheel-cache-aygt75b5/wheels/a6/d9/c0/26addf947458e4f1ac35ee3e7a31b4d71f7d6a2276da608f6e
Successfully built lerobot
  Attempting uninstall: lerobot
    Found existing installation: lerobot 0.4.1
    Uninstalling lerobot-0.4.1:
      Successfully uninstalled lerobot-0.4.1


## Auth HF
---
Make sure HF_TOKEN is set in colab secrets

In [3]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

## Config
---

In [4]:
HF_USER = "rlodhi"
DATASET_NAME = HF_USER + "/screw_pick_perfect"
MOUNTED_DATASET = "/root/.cache/huggingface/lerobot/" + DATASET_NAME
POLICY_NAME = HF_USER + "/smolvla_screw_picking_colab"

## Pull HF Dataset to .cache
---

In [7]:
from huggingface_hub import snapshot_download
import json

snapshot_download(
    repo_id=DATASET_NAME,
    repo_type="dataset",
    local_dir=MOUNTED_DATASET
)

print("✅ Dataset downloaded!")

#
# Verify
#

with open(f"{MOUNTED_DATASET}/meta/info.json") as f:
    info = json.load(f)

print("✅ info.json found!")
print(f"Episodes: {info['total_episodes']}")
print(f"Frames: {info['total_frames']}")
print(f"Cameras: {[k for k in info['features'] if 'images' in k]}")

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

✅ Dataset downloaded!
✅ info.json found!
Episodes: 100
Frames: 50965
Cameras: ['observation.images.base', 'observation.images.wrist']


## Training
---
This cell runs the `train.py` script from the `lerobot` library to train a robot control policy.  

Make sure to adjust the following arguments to your setup:

1. `--dataset.repo_id=YOUR_HF_USERNAME/YOUR_DATASET`:  
   Replace this with the Hugging Face Hub repo ID where your dataset is stored, e.g., `rlodhi/screw_pick_perfect`.

2. `--policy.type=act`:  
   Specifies the policy configuration to use. `act` refers to [configuration_act.py](../lerobot/common/policies/act/configuration_act.py), which will automatically adapt to your dataset's setup (e.g., number of motors and cameras).

3. `--output_dir=outputs/...`:  
   Directory where training logs and model checkpoints will be saved.
   
   _**IMPORTANT!** If you want to persist checkpoints and final model weights, simply mount your drive and point the output dir there. Ommitted here for simplicity._

4. `--job_name=...`:  
   A name for this training job, used for logging and Weights & Biases.

5. `--policy.device=cuda`:  
   Use `cuda` if training on an NVIDIA GPU. Use `mps` for Apple Silicon, or `cpu` if no GPU is available.

In [8]:
import torch
torch.cuda.is_available()

True

In [9]:
!nvidia-smi
# T4  → batch_size=8
# L4  → batch_size=16
# A100 → batch_size=32

Wed Apr 29 05:10:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
!lerobot-train \
--dataset.repo_id={DATASET_NAME} \
--policy.type=smolvla \
--output_dir={"./outputs/" + POLICY_NAME} \
--job_name=smolvla_training \
--policy.pretrained_path=lerobot/smolvla_base \
--policy.repo_id={POLICY_NAME} \
--steps=400 \
--policy.device=cuda \
--batch_size=8 \
--num_workers=2

2026-04-29 05:16:58.407820: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO 2026-04-29 05:17:06 ot_train.py:163 {'batch_size': 8,
 'checkpoint_path': None,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'R